# Phase 15: Privacy filter (local model with Ollama)

A citizen's report can hold a name, a phone number, an ID number or a home address. Before any of it goes to a cloud service (Gemini, Groq, LangSmith), a small model that runs **on this computer** removes the personal details.

Two tools do the work:
- **Regex** for details with a fixed shape: ID numbers, phone numbers, emails.
- **A local Ollama model** for names and street addresses. It only *lists* them. Our code does the replacing, so the model can never rewrite a report.

Start Ollama first (it runs in the background after install). Nothing in this notebook calls a cloud service.

In [ ]:
from unittest.mock import patch

from community_voice.privacy import redact as privacy
from community_voice.privacy.redact import find_pii, protect, redact, redact_patterns
from community_voice.schemas.privacy import PiiSpans

## 1. Regex: details with a fixed shape

ID numbers, phone numbers and emails are removed with patterns. No AI is needed, so this step is fast and always the same.

In [ ]:
text = (
    "My name is Thandi Mokoena, ID 9003045800086. There is a burst pipe outside "
    "27 Vilakazi Street, Soweto. Call me on 082 555 0142."
)
cleaned, counts = redact_patterns(text)
print(cleaned)
print(dict(counts))

## 2. The local model: names and addresses

The model gets the text that is left and lists the names and street addresses it finds. This runs on your computer, so the first call is slow while the model loads.

In [ ]:
spans = find_pii(cleaned)
print(spans)

## 3. Both together

`redact` runs the regex, asks the model, then replaces each listed span that really appears in the text. The area (Soweto) stays, because routing needs it.

In [ ]:
result = redact(text)
print(result.text)
print(result.counts)

## 4. Places are never removed

A small model sometimes lists a place by mistake, for example an area inside an address. `keep_places` drops known South African places (`data/knowledge_base/places.json`) and keeps only the address parts that have a number or a street word.

In [ ]:
from community_voice.privacy.redact import keep_places

guess = PiiSpans(
    names=["Mitchells Plain", "Sipho"], addresses=["12 Mandela Road, Umlazi"]
)
print(keep_places(guess))

## 5. When Ollama is not available

The filter **fails closed**: without the local model, the report is stopped and nothing is sent anywhere. A citizen can agree to continue. Then only the numbers and emails are removed, and the app says the local model was not used.

Here we pretend the model is down.

In [ ]:
with patch.object(
    privacy, "redact", side_effect=privacy.PrivacyFilterError("ConnectError")
):
    try:
        protect(text)
    except privacy.PrivacyFilterError as error:
        print("Stopped:", error)
    print(protect(text, allow_unfiltered=True))

## 6. How well does it work?

`data/evals/pii_reports.json` holds 12 fictional reports (English, isiZulu, isiXhosa, Afrikaans, Sesotho) with the personal details we planted. The score is the share of planted details that were removed, and whether the area was kept.

Run all 12 from the terminal (about two minutes):

```bash
uv run python -m evals.run_privacy_evals
```

Here are three of them.

In [ ]:
import json
import sys

from community_voice.core.config import PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT))  # so "evals" can be imported

from evals import privacy_scoring  # noqa: E402

reports = json.loads(
    (PROJECT_ROOT / "data" / "evals" / "pii_reports.json").read_text(encoding="utf-8")
)
rows = []
for report in reports[2:5]:
    redacted = redact(report["text"]).text
    print(report["language"], "|", redacted)
    rows.append(privacy_scoring.score_report(redacted, report["pii"], report["area"]))
print(privacy_scoring.summarize(rows))

## What to remember

| | Local model (Ollama) | Cloud model (Gemini, Groq) |
|---|---|---|
| Where the text goes | Stays on this computer | Leaves the computer |
| Speed | Slower here (about 7 to 12 seconds a report) | Fast |
| Quality | Weaker, especially with African-language names | Stronger |
| Cost | Free, uses your RAM and CPU | Free tier, with limits |

- Redaction lowers the risk. It does not make the data anonymous.
- A small local model misses some names. The eval measures this, and a person can still read the report in review.
- Voice notes are sent to Gemini for transcription **before** any redaction, because Ollama cannot transcribe speech. The transcript is redacted before the workflow starts.